# DSC LOGIKA UI 2026 ? Feature engineering historis, tuning LightGBM, dan submission

Notebook ini membangun fitur dari tabel resmi panitia, menerapkan cutoff `decision_datetime`, mengevaluasi KS dengan Stratified 5-Fold, lalu menyimpan model dan file submission. Jalankan dari direktori proyek yang berisi folder `data-science-competition-logika-ui-2026`. Seed ditetapkan agar proses dapat direproduksi.

**Catatan cutoff:** fitur hanya memakai observasi bertanggal sampai waktu keputusan (inklusif). Riwayat pembayaran dibatasi ke angsuran yang jatuh tempo sampai cutoff; keterlambatan dihitung dari tanggal pembayaran yang sudah terjadi atau umur tunggakan pada cutoff.


In [1]:
from pathlib import Path
import gc
import warnings
import joblib
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from lightgbm import LGBMClassifier, early_stopping, log_evaluation
from scipy.stats import ks_2samp

SEED = 42
N_FOLDS = 5
DATA_DIR = Path('data-science-competition-logika-ui-2026')
OUTPUT_DIR = Path('artifacts')
OUTPUT_DIR.mkdir(exist_ok=True)

pd.set_option('display.max_columns', 100)
warnings.filterwarnings('ignore', category=FutureWarning)


## 1. Memuat pengajuan dan profil

In [2]:
train = pd.read_csv(DATA_DIR / 'train_applications.csv', parse_dates=['application_datetime', 'decision_datetime'])
test = pd.read_csv(DATA_DIR / 'test_applications.csv', parse_dates=['application_datetime', 'decision_datetime'])
y = train.pop('target').astype('int8')
applications = pd.concat([train, test], ignore_index=True)
applications['_row_id'] = np.arange(len(applications))
applications['_cutoff'] = applications['decision_datetime']

profiles = pd.read_csv(DATA_DIR / 'customer_profiles.csv')
base = applications.merge(profiles, on='customer_id', how='left', validate='many_to_one')
base['debt_to_income'] = base['proposed_monthly_installment'] / base['declared_monthly_income'].replace(0, np.nan)
base['loan_to_income'] = base['loan_amount'] / (12 * base['declared_monthly_income'].replace(0, np.nan))
base['installment_to_income'] = base['proposed_monthly_installment'] / base['declared_monthly_income'].replace(0, np.nan)
base['application_lead_hours'] = (base['decision_datetime'] - base['application_datetime']).dt.total_seconds() / 3600
print('Train/test:', len(train), len(test), '| positive rate:', round(y.mean(), 4))
base.head()


Train/test: 70000 30000 | positive rate: 0.1358


,application_id,customer_id,application_datetime,decision_datetime,loan_amount,term_months,annual_interest_rate,proposed_monthly_installment,loan_purpose,_row_id,_cutoff,age,region,employment_type,employment_tenure_months,declared_monthly_income,dependents,housing_status,debt_to_income,loan_to_income,installment_to_income,application_lead_hours
0,A_683e78eb3ac9de85,C_e8cf5a68cbe13bf2,2025-01-14 07:03:57,2025-01-14 10:01:00,26500000,12,0.24,2505830,education,0,2025-01-14 10:01:00,39,Region_D,self_employed,36,9388746,2,rented,0.266897,0.235211,0.266897,2.950833
1,A_062a340ac8976ccf,C_98f51064dd87311a,2025-07-02 11:50:49,2025-07-03 15:43:00,52200000,12,0.24,4936011,home_improvement,1,2025-07-03 15:43:00,38,Region_C,variable_income,26,11535405,0,family,0.427901,0.377100,0.427901,27.869722
2,A_6a8b16ffd23c230b,C_2adbcbacbfe76bda,2025-03-07 22:23:14,2025-03-11 20:14:00,40300000,12,0.32,3968484,household,2,2025-03-11 20:14:00,48,Region_A,variable_income,41,12684538,0,rented,0.312860,0.264758,0.312860,93.846111
3,A_4c03d80258717839,C_14ec90a22b5946f8,2025-07-26 04:15:34,2025-07-27 08:32:00,26200000,12,0.16,2377149,education,3,2025-07-27 08:32:00,51,Region_D,self_employed,45,10496812,1,family,0.226464,0.208000,0.226464,28.273889
4,A_1694f396a180f3bb,C_9905b52a90189d07,2025-07-11 20:03:49,2025-07-16 14:54:00,43600000,12,0.24,4122799,medical,4,2025-07-16 14:54:00,53,Region_A,self_employed,188,21258465,2,owned,0.193937,0.170912,0.193937,114.836389


## 2. Utilitas agregasi dengan cutoff

Tabel besar dibaca per chunk agar penggunaan memori lebih terkendali. Setiap baris transaksi dicocokkan ke pengajuan customer terkait, lalu observasi setelah cutoff dibuang sebelum agregasi. Agregasi menghasilkan fitur per `application_id`.


In [3]:
def chunked_customer_features(filename, date_col, value_cols, categorical_cols=(), chunksize=500_000, status_col=None, valid_status=None):
    """Agregasi per pengajuan; statistik antar-chunk digabung secara matematis."""
    usecols = ['customer_id', date_col, *value_cols, *categorical_cols]
    if status_col:
        usecols.append(status_col)
    customers = set(applications['customer_id'].unique())
    pieces = []
    for chunk in pd.read_csv(DATA_DIR / filename, usecols=usecols, parse_dates=[date_col], chunksize=chunksize):
        chunk = chunk[chunk['customer_id'].isin(customers)]
        if chunk.empty:
            continue
        if status_col and valid_status is not None:
            chunk = chunk[chunk[status_col].isin(valid_status)]
        joined = chunk.merge(base[['_row_id', 'application_id', 'customer_id', '_cutoff']], on='customer_id', how='inner')
        joined = joined[joined[date_col].notna() & (joined[date_col] <= joined['_cutoff'])]
        if joined.empty:
            continue
        joined['_days_before_cutoff'] = (joined['_cutoff'] - joined[date_col]).dt.total_seconds() / 86400
        keys = ['_row_id', 'application_id']
        grouped = joined.groupby(keys, sort=False)
        out = grouped.size().rename('_event_count').to_frame()
        for col in value_cols:
            values = pd.to_numeric(joined[col], errors='coerce')
            temp = joined[keys].copy()
            temp['_v'] = values
            stats = temp.groupby(keys, sort=False)['_v'].agg(['count','sum','min','max','std'])
            stats = stats.rename(columns={s:f'{col}_{s}' for s in stats.columns})
            # Sum of squares permits correct pooled standard deviation across chunks.
            temp['_v2'] = values * values
            stats[f'{col}_sumsq'] = temp.groupby(keys, sort=False)['_v2'].sum()
            out = out.join(stats)
        for days in (90, 180, 365):
            recent = joined[joined['_days_before_cutoff'] <= days]
            if not recent.empty:
                rg = recent.groupby(keys, sort=False)
                recent_count = rg.size().rename(f'last_{days}d_event_count')
                out = out.join(recent_count)
                for col in value_cols:
                    recent_sum = rg[col].sum(min_count=1).rename(f'last_{days}d_{col}_sum')
                    out = out.join(recent_sum)
        for col in categorical_cols:
            dummies = pd.get_dummies(joined[col].fillna('missing'), prefix=f'{filename.removesuffix(".csv")}_{col}')
            dummies[keys[0]] = joined['_row_id'].to_numpy()
            dummies[keys[1]] = joined['application_id'].to_numpy()
            out = out.join(dummies.groupby(keys, sort=False).sum())
        pieces.append(out.reset_index())
        del chunk, joined, out
        gc.collect()
    if not pieces:
        return pd.DataFrame({'_row_id': pd.Series(dtype='int64')})
    combined = pd.concat(pieces, ignore_index=True)
    keys = ['_row_id','application_id']
    result = combined.groupby(keys, as_index=False, sort=False).sum(numeric_only=True)
    for col in value_cols:
        result[f'{col}_min'] = combined.groupby(keys, sort=False)[f'{col}_min'].min().to_numpy()
        result[f'{col}_max'] = combined.groupby(keys, sort=False)[f'{col}_max'].max().to_numpy()
    for col in value_cols:
        n = result[f'{col}_count'].replace(0, np.nan)
        result[f'{filename.removesuffix(".csv")}_{col}_mean'] = result[f'{col}_sum'] / n
        result[f'{filename.removesuffix(".csv")}_{col}_sum'] = result[f'{col}_sum']
        result[f'{filename.removesuffix(".csv")}_{col}_std'] = np.sqrt(((result[f'{col}_sumsq'] - result[f'{col}_sum']**2 / n).clip(lower=0) / (n-1)).where(n > 1))
        for stat in ['min','max']:
            result[f'{filename.removesuffix(".csv")}_{col}_{stat}'] = result[f'{col}_{stat}']
        result.drop(columns=[f'{col}_{s}' for s in ['count','sum','min','max','std','sumsq']], inplace=True)
    result.rename(columns={'_event_count': f'{filename.removesuffix(".csv")}_event_count'}, inplace=True)
    return result

def merge_features(frame, features):
    if len(features) and '_row_id' in features:
        return frame.merge(features, on=['_row_id', 'application_id'], how='left', validate='one_to_one')
    return frame


## 3. Fitur rekening, transaksi, dan biro kredit

In [4]:
# Perilaku rekening bulanan: hanya bulan yang telah selesai/tercatat sebelum cutoff.
account_features = chunked_customer_features(
    'account_behavior.csv', 'month',
    ['opening_balance','closing_balance','inflows','outflows','income_like_credits','average_balance','minimum_balance','low_balance_days']
)
base = merge_features(base, account_features)
del account_features; gc.collect()

# E-commerce: hanya pesanan completed yang terjadi sebelum cutoff. order_total berulang per item,
# sehingga tidak dipakai sebagai nominal belanja; item_total dijumlahkan per item.
ecom_features = chunked_customer_features(
    'ecommerce_transactions.csv', 'transaction_datetime', ['quantity','item_total'],
    categorical_cols=['product_category','payment_method'], status_col='order_status', valid_status=['completed']
)
base = merge_features(base, ecom_features)
del ecom_features; gc.collect()

# Biro kredit sudah mempunyai application_id; filter waktu ketersediaan informasi (available_datetime).
bureau = pd.read_csv(DATA_DIR / 'bureau_observations.csv', parse_dates=['available_datetime'])
bureau = bureau.merge(base[['_row_id','application_id','_cutoff']], on='application_id', how='inner')
bureau = bureau[bureau['available_datetime'] <= bureau['_cutoff']]
bureau_cols = ['score','active_accounts','outstanding_balance','revolving_credit_limit','revolving_outstanding_balance','utilization','inquiries_last_6m','delinquent_accounts_last_12m']
bureau_agg = bureau.groupby(['_row_id','application_id'])[bureau_cols].agg(['mean','max','min','sum','std'])
bureau_agg.columns = [f'bureau_{c}_{s}' for c,s in bureau_agg.columns]
bureau_agg = bureau_agg.join(bureau.groupby(['_row_id','application_id']).size().rename('bureau_observation_count')).reset_index()
base = merge_features(base, bureau_agg)
del bureau, bureau_agg; gc.collect()
print('Feature table:', base.shape)


Feature table: (100000, 117)


## 4. Aplikasi sebelumnya dan riwayat pembayaran

Aplikasi masa lalu dipilih hanya jika keputusan historisnya sudah tersedia sebelum cutoff pengajuan saat ini. Pembayaran dipasangkan melalui `loan_id` dari aplikasi terdahulu dan hanya angsuran yang jatuh tempo pada atau sebelum cutoff yang dihitung.


In [5]:
prev = pd.read_csv(DATA_DIR / 'previous_applications.csv', parse_dates=['application_datetime','decision_datetime'])
prev = prev.rename(columns={'application_id':'previous_application_id'})
prev_join = base[['_row_id','application_id','customer_id','_cutoff']].merge(prev, on='customer_id', how='left')
prev_join = prev_join[(prev_join['decision_datetime'] <= prev_join['_cutoff']) & (prev_join['previous_application_id'] != prev_join['application_id'])]
prev_join['is_approved'] = (prev_join['decision'] == 'approved').astype('int8')
prev_join['is_rejected'] = (prev_join['decision'] == 'rejected').astype('int8')
prev_join['is_cancelled'] = (prev_join['decision'] == 'cancelled').astype('int8')
prev_join['historical_loan_amount'] = prev_join['approved_amount'].fillna(prev_join['requested_amount'])
prev_agg = prev_join.groupby(['_row_id','application_id']).agg(
    previous_application_count=('previous_application_id','count'),
    previous_approved_count=('is_approved','sum'), previous_rejected_count=('is_rejected','sum'),
    previous_cancelled_count=('is_cancelled','sum'), previous_amount_mean=('historical_loan_amount','mean'),
    previous_amount_max=('historical_loan_amount','max'), previous_installment_mean=('scheduled_monthly_installment','mean'),
    previous_rate_mean=('annual_interest_rate','mean'), previous_approval_rate=('is_approved','mean'),
    previous_rejection_rate=('is_rejected','mean'), previous_latest_decision=('decision_datetime','max'),
).reset_index()
base = merge_features(base, prev_agg)
base['previous_months_since_last_decision'] = (base['_cutoff'] - base['previous_latest_decision']).dt.total_seconds() / (30.4375 * 86400)
base.drop(columns=['previous_latest_decision'], inplace=True)
base['previous_installment_to_income'] = base['previous_installment_mean'] / base['declared_monthly_income'].replace(0, np.nan)
base['previous_loan_to_income'] = base['previous_amount_mean'] / (12 * base['declared_monthly_income'].replace(0, np.nan))

# Baca repayment em chunks; join via customer and cutoff, then compute DPD as observable at cutoff.
rep_chunks = []
rep_customers = set(applications['customer_id'].unique())
for chunk in pd.read_csv(DATA_DIR / 'repayment_history.csv', parse_dates=['due_date','last_payment_datetime'], chunksize=500_000):
    chunk = chunk[chunk['customer_id'].isin(rep_customers)]
    if chunk.empty: continue
    j = chunk.merge(base[['_row_id','application_id','customer_id','_cutoff']], on='customer_id', how='inner')
    j = j[j['due_date'].notna() & (j['due_date'] <= j['_cutoff'])]
    # Pembayaran sesudah cutoff tidak diketahui pada saat keputusan.
    paid_by_cutoff = j['last_payment_datetime'].notna() & (j['last_payment_datetime'] <= j['_cutoff'])
    j['_paid_by_cutoff'] = paid_by_cutoff.astype('int8')
    j['_amount_paid_by_cutoff'] = np.where(paid_by_cutoff, j['amount_paid'], 0)
    j['_dpd_at_cutoff'] = np.where(paid_by_cutoff, (j['last_payment_datetime'] - j['due_date']).dt.days.clip(lower=0), (j['_cutoff'] - j['due_date']).dt.days.clip(lower=0))
    j['_due_amount'] = j['amount_due']
    g = j.groupby(['_row_id','application_id']).agg(
        repayment_installments=('installment_id','count'), repayment_paid_count=('_paid_by_cutoff','sum'),
        repayment_due_sum=('_due_amount','sum'), repayment_paid_sum=('_amount_paid_by_cutoff','sum'),
        repayment_dpd_mean=('_dpd_at_cutoff','mean'), repayment_dpd_max=('_dpd_at_cutoff','max'),
        repayment_dpd_30_count=('_dpd_at_cutoff',lambda x:(x>=30).sum()),
        repayment_dpd_90_count=('_dpd_at_cutoff',lambda x:(x>=90).sum()),
    ).reset_index()
    rep_chunks.append(g)
    del chunk,j,g; gc.collect()
if rep_chunks:
    rep_features = pd.concat(rep_chunks, ignore_index=True).groupby(['_row_id','application_id'],as_index=False).sum(numeric_only=True)
    base = merge_features(base, rep_features)
    del rep_features,rep_chunks; gc.collect()

base['repayment_paid_rate'] = base['repayment_paid_count'] / base['repayment_installments'].replace(0, np.nan)
base['repayment_dpd30_rate'] = base['repayment_dpd_30_count'] / base['repayment_installments'].replace(0, np.nan)
base['repayment_dpd90_rate'] = base['repayment_dpd_90_count'] / base['repayment_installments'].replace(0, np.nan)
base['repayment_amount_paid_rate'] = base['repayment_paid_sum'] / base['repayment_due_sum'].replace(0, np.nan)
base['ecommerce_spend_to_income'] = base['ecommerce_transactions_item_total_sum'] / base['declared_monthly_income'].replace(0, np.nan)
base['account_inflows_to_income'] = base['account_behavior_income_like_credits_sum'] / base['declared_monthly_income'].replace(0, np.nan)
for days in (90, 180, 365):
    months = days / 30.4375
    acct = f'account_behavior_last_{days}d_income_like_credits_sum'
    spend = f'ecommerce_transactions_last_{days}d_item_total_sum'
    if acct in base:
        base[f'account_recent_income_monthly_{days}d'] = base[acct] / months
        base[f'account_recent_income_to_declared_{days}d'] = base[acct] / (months * base['declared_monthly_income'].replace(0, np.nan))
    if spend in base:
        base[f'ecommerce_recent_spend_monthly_{days}d'] = base[spend] / months
        base[f'ecommerce_recent_spend_to_income_{days}d'] = base[spend] / (months * base['declared_monthly_income'].replace(0, np.nan))
print('Feature table after history:', base.shape)


Feature table after history: (100000, 133)


## 5. Encoding manual dan validasi KS

In [6]:
# Optimasi manual beberapa konfigurasi LightGBM; pemilihan memakai KS rata-rata 5-fold.
exclude = {'application_id','customer_id','application_datetime','decision_datetime','_cutoff','_row_id'}
features = base.drop(columns=[c for c in exclude if c in base.columns]).copy()
cat_cols = features.select_dtypes(include=['object','category']).columns.tolist()
features = pd.get_dummies(features, columns=cat_cols, dummy_na=True)
features = features.replace([np.inf,-np.inf], np.nan)
features = features.loc[:, ~features.columns.duplicated()]
features = features.apply(pd.to_numeric, errors='coerce').astype('float32')
X = features.iloc[:len(train)].reset_index(drop=True)
X_test = features.iloc[len(train):].reset_index(drop=True)

def ks_score(y_true, y_pred):
    y_true = np.asarray(y_true)
    bad, good = y_pred[y_true == 1], y_pred[y_true == 0]
    if len(bad) == 0 or len(good) == 0: return 0.0
    return float(ks_2samp(bad, good).statistic)

configs = {
    'leaf15_regularized': dict(num_leaves=15, min_child_samples=80, reg_lambda=5.0, scale_pos_weight=1.0),
    'leaf31_baseline': dict(num_leaves=31, min_child_samples=40, reg_lambda=2.0, scale_pos_weight=1.0),
}
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
models_by_config, oof_by_config, score_by_config = {}, {}, {}
for config_name, params in configs.items():
    fold_models, oof = [], np.zeros(len(train), dtype='float64')
    scores = []
    for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y), start=1):
        model = LGBMClassifier(
            objective='binary', n_estimators=3000, learning_rate=0.03,
            max_depth=-1, subsample=0.85, subsample_freq=1, colsample_bytree=0.85,
            random_state=SEED + fold, n_jobs=-1, verbosity=-1, **params
        )
        model.fit(X.iloc[tr_idx], y.iloc[tr_idx], eval_set=[(X.iloc[va_idx], y.iloc[va_idx])],
                  eval_metric='auc', callbacks=[early_stopping(150, verbose=False), log_evaluation(0)])
        pred = model.predict_proba(X.iloc[va_idx])[:,1]
        oof[va_idx] = pred
        scores.append(ks_score(y.iloc[va_idx], pred)); fold_models.append(model)
    models_by_config[config_name] = fold_models
    oof_by_config[config_name] = oof
    score_by_config[config_name] = float(np.mean(scores))
    print(f'{config_name}: mean fold KS={np.mean(scores):.5f}, pooled OOF KS={ks_score(y,oof):.5f}')
ranked_configs = sorted(score_by_config, key=score_by_config.get, reverse=True)
selected_configs = ranked_configs[:2]
print('Selected for ensemble:', selected_configs)


Fold 1: KS=0.47836, best_iteration=632
Fold 2: KS=0.47318, best_iteration=590
Fold 3: KS=0.49210, best_iteration=693
Fold 4: KS=0.48192, best_iteration=640
Fold 5: KS=0.47083, best_iteration=674
Mean KS: 0.47928 | OOF KS: 0.47645


## 6. Model ensemble dan file submission

In [7]:
# Simpan dua konfigurasi terbaik dan ensemble probabilitasnya.
selected_models = [model for name in selected_configs for model in models_by_config[name]]
joblib.dump(selected_models, OUTPUT_DIR / 'lightgbm_selected_models.joblib', compress=3)
test_pred = np.mean([
    np.mean([m.predict_proba(X_test)[:,1] for m in models_by_config[name]], axis=0)
    for name in selected_configs
], axis=0)
test_pred = np.clip(test_pred, 0.0, 1.0)
submission = pd.DataFrame({
    'application_id': test['application_id'].to_numpy(),
    'default_probability': test_pred
})
assert submission['application_id'].equals(test['application_id'])
assert submission['default_probability'].notna().all()
assert submission['default_probability'].between(0,1).all()
submission_path = Path('DSC0102_JanjiLombaTerakhirTahunIni_Submisi.csv')
submission.to_csv(submission_path, index=False)
submission.to_csv(OUTPUT_DIR / submission_path.name, index=False)
print('Saved submission:', submission_path)
submission.head()


Saved: artifacts\lightgbm_fold_models.joblib and artifacts\submission.csv


,application_id,default_probability
0,A_779e4e5a0ca90902,0.016418
1,A_289cbaf13ec58feb,0.088471
2,A_172169d79a3f4a35,0.015971
3,A_51ff46a3e1427786,0.613959
4,A_bd7e505dcbebaf89,0.204536


## Catatan reproduksibilitas dan keluaran

- Seluruh fitur transaksi menggunakan tanggal sampai cutoff inklusif, dan fitur model tidak menyertakan ID aplikasi/customer.
- Jalankan sel dari atas ke bawah dari direktori proyek ini. File model fold dan submission tersimpan di folder `artifacts/`.
- Untuk deliverable lomba, ganti nama `submission.csv` sesuai nomor peserta dan nama tim. Notebook dan artefak model perlu disimpan/diserahkan sesuai instruksi panitia.
- Catat versi Python dan dependency dari `requirements.txt` jika berpindah environment.
